### RAG v1


In [23]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('chatbot_rag_v1') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [62]:
# !pip install pyspark langchain
# !pip install langchain_community

In [103]:
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import ChatOllama

from dotenv import load_dotenv
import os
from pyspark.sql.types import StructType
from pyspark.sql import DataFrame

In [25]:
%run ./01_Config_env.ipynb

python-dotenv já está instalado.


In [81]:
print(OLLAMA_API_URL)

http://ollama:11434


In [82]:

OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
token = os.getenv("API_TOKEN")

In [27]:
 %run ./02_Common.ipynb

In [37]:
%run ./03_Get_data.ipynb

Dados disponivei: 

root
 |-- c: string (nullable = true)
 |-- cl: string (nullable = true)
 |-- sl: string (nullable = true)
 |-- lt0: string (nullable = true)
 |-- lt1: string (nullable = true)
 |-- qv: string (nullable = true)
 |-- vs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- p: string (nullable = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- ta: string (nullable = true)
 |    |    |-- py: string (nullable = true)
 |    |    |-- px: string (nullable = true)
 |    |    |-- sv: string (nullable = true)
 |    |    |-- is: string (nullable = true)



### Pegar uma amostra dos dados

In [119]:
df_posicao.show(10)

+-------+-----+---+----------------+-----------------+---+--------------------+
|      c|   cl| sl|             lt0|              lt1| qv|                  vs|
+-------+-----+---+----------------+-----------------+---+--------------------+
|5103-10| 1646|  1|           MOEMA|     TERM. SACOMÃ|  4|[{52846, true, 20...|
|6030-10|33910|  2|TERM. STO. AMARO|   UNISA-CAMPUS 1|  8|[{66143, true, 20...|
|6021-10|  177|  1|    JD. NOVA ERA|   TERM. VARGINHA|  3|[{66305, true, 20...|
|271F-10|33451|  2|METRÔ BELÉM     |     CENTER NORTE|  6|[{21742, true, 20...|
|857A-10|34693|  2| METRÔ STA. CRUZ|TERM. CAMPO LIMPO|  7|[{75600, true, 20...|
|6840-10|  201|  1| TERM. CAPELINHA| TERM. JD. JACIRA|  5|[{73614, true, 20...|
|6262-10| 1371|  1|  TERM. BANDEIRA|            CEASA|  5|[{81348, true, 20...|
|573H-10| 1630|  1|   METRÔ BRESSER|  HOSP. SAPOPEMBA| 13|[{55122, true, 20...|
|6L10-10| 1159|  1|PQ. FLORESTAL   |   TERM. VARGINHA|  2|[{66427, true, 20...|
|407W-10|34972|  2|    METRÔ CARRÃO|JD. 

In [121]:
from pyspark.sql.functions import explode, col

df = df_posicao.select(
    col('c').alias('Letreiro_Linha'),
    col('cl').alias('Linha'),
    col('sl').alias('Sentido'),
    col('lt0').alias('Destino_Linha'),
    col('lt1').alias('Origem_Linha'),
    col('qv').cast('int').alias('Quantidade_Veiculos')
    
).limit(10)

df.createOrReplaceTempView("tbl_bus_posicao")

### Carrega modelo (Mistral 7B)

In [79]:
llm = ChatOllama(model="mistral:latest", base_url=OLLAMA_API_URL) 

### Configurar Promps (roles System, Human)

In [112]:
# Prompt para gerar SQL (com roles)
prompt_sql = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um especialista em dados. Gere apenas a consulta SQL."),
    HumanMessagePromptTemplate.from_template(
        "Com base na estrutura da tabela abaixo:\n\n{schema}\n\n"
        "Escreva uma consulta SQL (somente a SQL) para responder:\n{pergunta}"
    )
])


In [113]:
# Prompt para gerar resposta para o usuário (com roles)
prompt_resposta = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um assistente de dados."),
    HumanMessagePromptTemplate.from_template(
        "Pergunta: {pergunta}\n\nResultado da consulta:\n{resultado}\n\n"
        "Gere uma resposta clara e amigável para o usuário."
    )
])

### Funções Auxiliares

In [114]:
def describe_table(df, table_name="tbl_bus_posicao"):
    colunas = "\n".join([f"- {f.name}: {f.dataType.simpleString()}" for f in df.schema])
    return f"Tabela: {table_name}\n\nColunas:\n{colunas}"


In [155]:
# Remover formatação do código gerado
import re

def limpar_sql(resposta_modelo):
    # Remove blocos de código markdown e espaços extras
    sql = re.sub(r"```sql|```", "", resposta_modelo, flags=re.IGNORECASE).strip()
    return sql

### Chatbot com RAG

In [116]:
def augmented_response(pergunta):
    print(f"Pergunta: {pergunta}")

    # Etapa 1: Gerar SQL com role
    schema_txt = describe_table(df, "tbl_bus_posicao")
    sql_chain = prompt_sql | llm
    sql_result = sql_chain.invoke({"pergunta": pergunta, "schema": schema_txt})
    sql_query = limpar_sql(sql_result.content)
    print(f"\n🤖💡 SQL Gerado:\n{sql_query}")

    # Etapa 2: Executar SQL
    try:
        resultado_df = spark.sql(sql_query).toPandas().to_dict(orient="records")
    except Exception as e:
        print(f"❌ Erro na execução da SQL: {e}")
        return

    # Etapa 3: Gerar resposta final com role
    resposta_chain = prompt_resposta | llm
    resposta_result = resposta_chain.invoke({
        "pergunta": pergunta,
        "resultado": resultado_df
    })
    resposta = resposta_result.content.strip()

    print(f"\n🤖 Resposta:\n{resposta}")

In [122]:
augmented_response("Qual o letreiro da linha (Letreiro_Linha) do veiculo que vai para o METRÔ STA. CRUZ?")

Pergunta: Qual o letreiro da linha (Letreiro_Linha) do veiculo que vai para o METRÔ STA. CRUZ?

🤖💡 SQL Gerado:
SELECT Letreiro_Linha
   FROM tbl_bus_posicao
   WHERE Destino_Linha = 'METRÔ STA. CRUZ';

🤖 Resposta:
O veículo que vai para a estação Metrô Sta. Cruz tem a seguinte legenda na linha: 857A-10.
